In [0]:
import requests
import uuid
import json

# 1. Obtener dinámicamente el host y el token desde el contexto de Databricks
context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = context.apiToken().get()
host = context.apiUrl().get()

# 2. Asegúrate de que el nombre coincida con tu endpoint
endpoint_name = "metabuilder-endpoint" 
invoke_url = f"{host}/serving-endpoints/{endpoint_name}/invocations"

# 3. Generar un thread_id único para la sesión
test_thread_id = str(uuid.uuid4())
print(f"Thread ID: {test_thread_id}")

In [0]:
payload = {
    "messages": [
        {"role": "user", "content": "udv_desa.sch_udv_vw.ud_poliza_cert_cobro_reaseg_gen_core"}
    ],
    "custom_inputs": {
        "thread_id": test_thread_id
    }
}

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
    "Accept": "text/event-stream"
}

print("Iniciando solicitud HTTP (Streaming)...")
response = requests.post(invoke_url, json=payload, headers=headers, stream=True)

if response.status_code != 200:
    print(f"❌ Error {response.status_code}: {response.text}")
else:
    print("✅ Conectado exitosamente. Recibiendo stream:\n")
    print("-" * 60)
    
    for chunk in response.iter_lines(decode_unicode=True):
        if chunk:
            try:
                # Intenta parsear el JSON y extraer los mensajes
                mensajes = json.loads(chunk)
                for msg in mensajes:
                    print(msg.strip())
                    print()
            except json.JSONDecodeError:
                # Respaldo por si llega un fragmento de texto puro
                print(chunk)
                
    print("-" * 60)
    print("\n✅ Flujo Inicial Completado.")